### Step 1: Load the Silver Orders dataset

In [0]:
silver_df = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

### Step 2: Check whether AQE is enabled

In [0]:
print(
    "AQE enabled: ",
    spark.conf.get("spark.sql.adaptive.enabled")
)

AQE enabled:  true (not accessible on Serverless, enabled by default)


In [0]:
from pyspark.sql.functions import sum,count

state_summary_df = (
    silver_df
    .groupBy("shipping_state")
    .agg(
        sum("final_amount").alias("total_amount"),
        count("order_id").alias("total_orders")
    )
)

display(state_summary_df)

shipping_state,total_amount,total_orders
Delhi,1508859.97,51
Uttar Pradesh,1895295.0999999996,67
Tamil Nadu,6789427.020000001,164
Rajasthan,3035472.2099999995,62
Telangana,3457330.1799999997,68
Gujarat,3023906.42,67
Kerala,1988542.8099999996,50
Maharashtra,5736330.220000002,101
West Bengal,1781345.2699999996,47
Karnataka,2587097.08,61


### Step 4: Inspect the initial execution plan

In [0]:
state_summary_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Initial Plan ==
   PhotonResultStage (8)
   +- PhotonColumnarToRow (7)
      +- PhotonGroupingAgg (6)
         +- PhotonShuffleExchangeSource (5)
            +- PhotonShuffleMapStage (4)
               +- PhotonShuffleExchangeSink (3)
                  +- PhotonGroupingAgg (2)
                     +- PhotonScan parquet workspace.ecommerce_project.silver_orders (1)


(1) PhotonScan parquet workspace.ecommerce_project.silver_orders
Output [3]: [order_id#10642, final_amount#10653, shipping_state#10657]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-kfvj9/uc/14e4159b-99f2-4385-8229-38bcd1d5fcc4/598f6ee0-bdbe-4ff2-b5d0-c5f3c7a6b2ff/__unitystorage/catalogs/ac49d258-c63d-4b32-b612-fb799cc225af/tables/7134db69-b24b-4b6b-b931-aec642cf9171]
ReadSchema: struct<order_id:string,final_amount:double,shipping_state:string>

(2) PhotonGroupingAgg
Input [3]: [order_id#10642, final_amount#10653, shipping_state#10657]
Arguments: [shipping_state#10657]

### Step 5: Execute the query and inspect runtime details

In [0]:
state_summary_df.collect()

[Row(shipping_state='Delhi', total_amount=1508859.97, total_orders=51),
 Row(shipping_state='Uttar Pradesh', total_amount=1895295.0999999996, total_orders=67),
 Row(shipping_state='Tamil Nadu', total_amount=6789427.020000001, total_orders=164),
 Row(shipping_state='Rajasthan', total_amount=3035472.2099999995, total_orders=62),
 Row(shipping_state='Telangana', total_amount=3457330.1799999997, total_orders=68),
 Row(shipping_state='Gujarat', total_amount=3023906.42, total_orders=67),
 Row(shipping_state='Kerala', total_amount=1988542.8099999996, total_orders=50),
 Row(shipping_state='Maharashtra', total_amount=5736330.220000002, total_orders=101),
 Row(shipping_state='West Bengal', total_amount=1781345.2699999996, total_orders=47),
 Row(shipping_state='Karnataka', total_amount=2587097.08, total_orders=61)]